Cấu hình — chỉ đọc snapshot; không gọi API.

In [ ]:
from pathlib import Path
import sys, platform, json
import pandas as pd
here = Path.cwd().resolve()
DATA_PREPARATION_DIR = next((p if p.name == 'data-preparation' else p / 'data-preparation' for p in [here, *here.parents] if (p if p.name == 'data-preparation' else p / 'data-preparation').is_dir()), None)
if DATA_PREPARATION_DIR is None:
    raise RuntimeError('Mở kernel trong repo hoặc đặt DATA_PREPARATION_DIR')
sys.path.insert(0, str(DATA_PREPARATION_DIR / 'scripts'))
import base_catalogue as bc
SNAPSHOTS = DATA_PREPARATION_DIR / 'snapshots'
REPORTS_DIR = DATA_PREPARATION_DIR / 'datasets/thu-duc-base/reports'
print(platform.python_version(), pd.__version__, SNAPSHOTS)


Đọc nguồn — xem kích thước, tên cột và mẫu.

In [ ]:
source = SNAPSHOTS / 'thu_duc_dishes_taxonomy.csv'
raw = bc.read_csv(source)
frame = pd.DataFrame(raw)
print(source.stat().st_size, frame.shape, list(frame.columns))
display(frame[['dish_id','dish_name','taxonomy_status']].head(10))


Xem quy tắc — bảng ID đã được chủ dự án chốt; tên lạ phải rà.

In [ ]:
rules = bc.load_json(DATA_PREPARATION_DIR / 'config/base_dish_mapping_v1.json')
registry = bc.load_json(DATA_PREPARATION_DIR / 'config/base_dish_registry.json')
codebook = bc.load_json(DATA_PREPARATION_DIR / 'config/taxonomy_v1.json')
display(pd.DataFrame(registry['dishes']))
display(pd.DataFrame(rules['mapping']))
print(json.dumps(codebook, ensure_ascii=False, indent=2))


Mapping — không gộp tự động theo tiền tố; hiển thị các trạng thái.

In [ ]:
base_rows, mapping = bc.build(raw, rules, registry)
map_frame = pd.DataFrame(mapping)
display(map_frame.groupby('status').size().rename('count').to_frame())
display(map_frame[map_frame.status != 'mapped'])


Tổng hợp — các đề xuất GPT giữ theo ngữ cảnh nguồn; món gốc chưa review.

In [ ]:
display(pd.DataFrame(base_rows)[['name','sourceNameCount','menuItemCount','classificationStatus']])
display(pd.DataFrame(base_rows[0]['menuOfferings']))
display(pd.DataFrame(base_rows[0]['sourceProfiles']))


Kiểm tra — 216 mapping, 2 cần xem lại, 1 loại; 39 món gốc.

In [ ]:
from collections import Counter
assert len(raw) == 219 and len(base_rows) == 39
assert Counter(m['status'] for m in mapping) == {'mapped':216,'needs_review':2,'excluded':1}
source_offerings = {o['menu_item_id']:o for r in raw for o in json.loads(r['menu_offerings_json'])}
for b in base_rows:
    for offering in b['menuOfferings']:
        assert offering == source_offerings[offering['menu_item_id']]
print('Dữ liệu offering được bảo toàn; không chuyển quyền ảnh/lịch/giá thành verified.')


Xuất — UTF-8 BOM; đọc lại kiểm từng giá trị; checksum ổn định.

In [ ]:
stats = bc.export(base_rows, mapping, SNAPSHOTS)
print(json.dumps(stats, ensure_ascii=False, indent=2))
display(pd.DataFrame(bc.read_csv(SNAPSHOTS / 'thu_duc_base_dishes.csv'))[['id','name','sourceNameCount']])
